In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import time

# 1. https://www.hasamdongcoffee.com/store.php의 url에 접속
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)
url = "https://www.hasamdongcoffee.com/store.php"
driver.get(url)
time.sleep(5)  # 페이지 및 리스트 로딩 대기

all_data = []

try:
    # 2. "placesList"라는 아이디명이 있는 html 요소 찾기 (보내주신 HTML 기준)
    # 7. 무한 스크롤(또는 긴 리스트) 대응을 위해 순차적으로 접근
    processed_seqs = set() # 중복 수집 방지

    while True:
        # 현재 브라우저에 로드된 item 클래스들 찾기
        items = driver.find_elements(By.CSS_SELECTOR, "#placesList .item")
        
        # 새로 수집할 아이템이 있는지 확인
        new_items = [el for el in items if el.get_attribute('seq') not in processed_seqs]
        
        if not new_items:
            # 더 이상 새로 나타나는 매장이 없으면 종료
            break

        for element in new_items:
            # 7. 해당 지점으로 스크롤 내리기 (하나하나씩 보이게 함)
            driver.execute_script("arguments[0].scrollIntoView();", element)
            time.sleep(0.1) # 짧은 로딩 대기

            # BeautifulSoup 파싱을 위해 현재 요소의 HTML 가져오기
            item_html = element.get_attribute('outerHTML')
            soup = BeautifulSoup(item_html, 'html.parser')
            item_tag = soup.find('li', class_='item')

            if item_tag:
                # 3. item 클래스 내 x값(위도)과 y값(경도) 수집
                lat = item_tag.get('x') # 위도
                lon = item_tag.get('y') # 경도

                # 4. item 클래스명 안에 info 클래스 접근
                info_div = item_tag.find('div', class_='info')
                
                if info_div:
                    # 5. nm 클래스명에서 텍스트 파싱 (매장명 컬럼)
                    name = info_div.find('h5', class_='nm').get_text(strip=True)
                    
                    # 6. addr 클래스명에서 텍스트 파싱 (주소 컬럼)
                    address = info_div.find('span', class_='addr').get_text(strip=True)

                    # 8. 매장명, 주소, 위도, 경도 순서로 데이터 구성
                    store_info = [name, address, lat, lon]
                    all_data.append(store_info)

                    # 중복 방지를 위한 seq 저장
                    processed_seqs.add(item_tag.get('seq'))

                    # 9. 지점 하나 끝날 때마다 콘솔로 띄워주기
                    print(f"수집 완료: {name} | {address}")

        # 스크롤 후 새로운 목록이 갱신될 시간을 줌
        time.sleep(1)

    # 8. 수집된 데이터를 데이터프레임으로 변환
    df = pd.DataFrame(all_data, columns=["매장명", "주소", "위도", "경도"])

    # 9. hasamdongcoffee.csv 파일로 저장
    df.to_csv("hasamdongcoffee.csv", index=False, encoding="utf-8-sig")
    print("\n" + "="*50)
    print(f"최종 수집 완료! 총 {len(df)}개 매장이 'hasamdongcoffee.csv'에 저장되었습니다.")
    print("="*50)

except Exception as e:
    print(f"오류 발생: {e}")

finally:
    driver.quit()

수집 완료: 부산국제물류산단점 | 부산 강서구 가락대로 699 (생곡동) 1층 일부호(생곡동)
수집 완료: 다산현대캠퍼스몰점 | 경기 남양주시 다산순환로 20 (다산동) 다산현대프리미어캠퍼스 1층 제디씨01-020호
수집 완료: 서울 잠실르엘점 | 서울 송파구 올림픽로 329 (신천동) B1층 117
수집 완료: 거제장목마을점 | 경남 거제시 장목면 거제북로 1163-1 (장목리) 우측 하삼동커피
수집 완료: 양산서창점 | 경남 양산시 서창서2길 2 (삼호동, 새마을금고) 1층 2-1호
수집 완료: 창녕대신새길점 | 경남 창녕군 남지읍 대신길 1 (남지리) 1층 하삼동커피
수집 완료: 인천영종운서역점 | 인천 중구 운서동 2803-1 메가스타 영종 2층 217호
수집 완료: 수원 경희대국제캠퍼스점 | 경기 수원시 영통구 영통동 1023-5 1층 101호
수집 완료: 창녕영산점 | 경남 창녕군 영산면 서리상촌길 321 (서리) 하삼동커피
수집 완료: 김해대동산단점 | 경남 김해시 대동면 대동산단2로 106 (월촌리) 1층
수집 완료: 부산가내찬태광후지킨점 | 부산 강서구 화전산단3로 32 (화전동) 24동 1층
수집 완료: 울산대송점 | 울산 동구 학문로 87 (화정동) 1층
수집 완료: 울산선암대나리점 | 울산 남구 두왕로190번길 18-1 (선암동) 1층
수집 완료: 시흥은계호수공원점 | 경기 시흥시 은계중앙로 250 (은행동, 은계호수타운2) 은계호수타운2 116, 117호
수집 완료: 울산태화국가정원 | 울산 중구 태화강국가정원길 55 (태화동) 1층 하삼동커피
수집 완료: 부산기장월전점 | 부산 기장군 기장읍 죽성로 378 (죽성리, 월전슈퍼) 1층 하삼동커피
수집 완료: 통영중앙시장점 | 경남 통영시 통영해안로 341 (중앙동) 1층 하삼동커피
수집 완료: 인천청라국제도시점 | 인천 서구 미래로 11 (청라동, 청라썬앤빌 에코스타) 청라썬앤빌에코스타 상가 104호
수집 완료: 김해가동마을점 | 경남 김해시 한림면 한림로816번길 66 (가동리) 2동 

In [6]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. CSV 파일 로드
file_path = 'paikdabang.csv'
df = pd.read_csv(file_path)

# 2. MySQL 연결 설정 (사용자 정보에 맞게 수정 필요)
user = 'root'      # MySQL 사용자명 (예: root)
password = '1234'  # MySQL 비밀번호
host = 'localhost'          # 호스트 (예: 127.0.0.1)
port = '3306'               # 포트 번호
db_name = 'coffee_store'    # 대상 데이터베이스

# 3. 데이터베이스 생성 (이미 존재하면 무시)
# 먼저 MySQL 서버 자체에 연결하여 DB가 없으면 생성합니다.
temp_engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}")
with temp_engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {db_name}"))
    print(f"데이터베이스 '{db_name}'가 준비되었습니다.")

# 4. SQLAlchemy 엔진 생성 (coffee_store DB 연결)
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}?charset=utf8mb4")

# 5. 데이터프레임을 MySQL 테이블로 저장
# - name: 생성할 테이블 이름 ('the_venti')
# - if_exists: 'fail' (이미 있으면 오류), 'replace' (기존 삭제 후 생성), 'append' (기존 테이블에 추가)
# - index: False (데이터프레임의 인덱스는 컬럼으로 넣지 않음)
try:
    df.to_sql(name='paikdabang', con=engine, if_exists='replace', index=False)
    print(f"테이블 'paikdabang'에 총 {len(df)}건의 데이터가 성공적으로 저장되었습니다.")
except Exception as e:
    print(f"데이터 저장 중 오류 발생: {e}")

# 연결 종료
engine.dispose()

데이터베이스 'coffee_store'가 준비되었습니다.
테이블 'paikdabang'에 총 1856건의 데이터가 성공적으로 저장되었습니다.
